# 02b - Bootstrap and Resampling

## From an observed difference to uncertainty

Lecture 01b described a difference in median grocery spending. Lecture 02a asked whether that difference would be unusual under a no-association model. This notebook asks a different question: if the recorded clients reasonably represent the populations of interest, how much would the estimated median difference vary across new samples like these?

The bootstrap answers this question by treating the observed data as an empirical stand-in for the unknown population distributions. It then recomputes the estimate on many samples drawn with replacement.

### Learning goals

By the end of this notebook, you should be able to:

- distinguish a point estimate, bootstrap replicate, bootstrap distribution, and confidence interval;
- distinguish sampling with replacement from sampling without replacement;
- explain the empirical-distribution assumption behind a bootstrap;
- explain how a pseudorandom number generator and seed support reproducible resampling;
- construct and interpret a percentile bootstrap interval; and
- identify structure and selection that a resampling procedure must preserve or repeat.

## Relevant course readings

No textbook is required for this meeting. These portions of the course references provide the closest companions to the notebook:

- Bruce, Bruce, and Gedeck, *Practical Statistics for Data Scientists*, second edition, Chapter 2: focus on “Sampling Distribution of a Statistic,” “The Bootstrap,” and “Confidence Intervals.” These sections connect repeated samples, standard errors, replacement, and interval construction.
- James et al., [*An Introduction to Statistical Learning with Applications in Python*](https://www.statlearning.com/), Section 5.2: “The Bootstrap,” especially Figures 5.10 and 5.11. The figures contrast inaccessible repeated sampling from a population with the bootstrap approximation and make duplicated and omitted observations visible. Section 5.3.3 shows the corresponding Python lab.
- Tan et al., *Introduction to Data Mining*, Chapter 10: “Avoiding False Discoveries.” This chapter is most relevant to the final section on searching several results and reporting the strongest one; it is not needed for the core bootstrap procedure.

## The Wholesale Customers case

The UCI *Wholesale customers* data set contains annual spending records for 440 clients of a wholesale distributor. `Channel` records an existing Horeca or Retail classification. `Region` records a geographic category, and `Grocery` records annual spending in source monetary units.

Data source: Cardoso, M. (2013). *Wholesale customers* [Dataset]. UCI Machine Learning Repository. <https://doi.org/10.24432/C5030X>. The source archive is available under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

In [ ]:
# Load the tabular, numerical, plotting, and resampling tools used below.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import bootstrap

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/292/wholesale%2Bcustomers.zip"

# pandas can read the CSV directly from the compressed source archive.
customers = pd.read_csv(DATA_URL, compression="zip")
customers["channel"] = customers["Channel"].map({1: "Horeca", 2: "Retail"})

# Separate arrays make the two-sample statistic and later resampling explicit.
retail_grocery = customers.loc[customers["channel"] == "Retail", "Grocery"].to_numpy()
horeca_grocery = customers.loc[customers["channel"] == "Horeca", "Grocery"].to_numpy()


def median_difference(x, y, axis=-1):
    # Keeping axis configurable lets SciPy evaluate many resamples at once.
    return np.median(x, axis=axis) - np.median(y, axis=axis)


observed_difference = median_difference(retail_grocery, horeca_grocery)

customers[["channel", "Region", "Grocery"]].head()

## Lecture 02a follow-up: preserve recorded structure

The supplied data describes each row as one anonymous client and provides no timestamp, year, repeated-client identifier, or recorded ordering variable. There is therefore no observed time order for the permutation to preserve. The absence of a timestamp does not prove that collection order never mattered; it means the supplied table gives no way to identify or preserve it.

`Region` is recorded structure that can be investigated. The table below shows that every region contains clients from both channels, but the channel counts differ across regions.

In [ ]:
# A cross-tabulation exposes the joint counts that a structured test may preserve.
region_channel_counts = pd.crosstab(customers["Region"], customers["channel"])
region_channel_counts

**Stratification** divides observations into predefined groups, or strata, and performs randomization or resampling separately within those groups so that their relevant structure is preserved.

Lecture 02a reassigned the existing `Channel` labels across all 440 clients while preserving 142 Retail and 298 Horeca labels. It therefore preserved the overall channel proportions, but not the Retail/Horeca composition within each `Region`. A region-stratified permutation reassigns `Channel` labels only among clients in the same region, preserving the channel counts in all three regions.

### Pause: predict the regional result

Do you expect preserving the recorded regional composition to overturn the Lecture 02a result? State what the channel-count table can and cannot tell you before running the test.

##### Answer

The counts show that region and channel composition are not identical, so region is worth preserving in a sensitivity check. The counts alone cannot show whether the median difference will remain unusual. That requires recomputing the permutation distribution under the region-stratified procedure.

In [ ]:
def region_stratified_permutation_distribution(data, n_resamples, rng):
    # Convert once outside the loop; positions connect each region to its rows.
    labels = data["channel"].to_numpy()
    grocery = data["Grocery"].to_numpy()
    region_positions = [
        np.flatnonzero(data["Region"].to_numpy() == region)
        for region in sorted(data["Region"].unique())
    ]

    differences = np.empty(n_resamples)
    for replicate in range(n_resamples):
        # Copy the labels, then shuffle only within each recorded region.
        reassigned_labels = labels.copy()
        for positions in region_positions:
            reassigned_labels[positions] = rng.permutation(labels[positions])

        # Re-form the two channel groups after reassignment and save the statistic.
        differences[replicate] = median_difference(
            grocery[reassigned_labels == "Retail"],
            grocery[reassigned_labels == "Horeca"],
        )

    return differences


N_REGION_PERMUTATIONS = 5_000
region_null_differences = region_stratified_permutation_distribution(
    customers,
    n_resamples=N_REGION_PERMUTATIONS,
    rng=np.random.default_rng(7130),
)

# The +1 adjustment includes the observed arrangement in the simulated reference set.
upper_tail = (
    np.count_nonzero(region_null_differences >= observed_difference) + 1
) / (N_REGION_PERMUTATIONS + 1)
lower_tail = (
    np.count_nonzero(region_null_differences <= observed_difference) + 1
) / (N_REGION_PERMUTATIONS + 1)
region_stratified_pvalue = min(1.0, 2 * min(upper_tail, lower_tail))

pd.Series(
    {
        "Observed median difference": observed_difference,
        "Simulated differences at least as extreme": np.count_nonzero(
            np.abs(region_null_differences) >= abs(observed_difference)
        ),
        "Region-stratified simulated p-value": region_stratified_pvalue,
    }
)

None of the 5,000 region-stratified reassignments produces a median difference as extreme as 9,706. Preserving the recorded regional composition does not materially change the Lecture 02a conclusion: the observed association remains very unusual under the more structured no-association model.

The region-stratified simulated p-value is `0.0004`, exactly the value reported for the overall permutation in Lecture 02a. The matching values do not mean that the two permutation distributions are identical. Both procedures produced zero reassignments in the relevant observed tail. They therefore reached the smallest two-sided p-value available from this calculation with 5,000 permutations:

$$
2 \times \frac{0 + 1}{5{,}000 + 1} \approx 0.0004.
$$

The shared value is a **simulation-resolution floor** created by the resample count and the `+1` correction. At this resolution, both procedures show that the observed result lies beyond every simulated value in the relevant tail; they do not reveal whether one underlying tail probability is smaller than the other. More permutations would provide finer resolution, although they would not change the substantive conclusion here.

Lecture 02a left one other question open. If an analyst searches several statistics or thresholds and reports the strongest result, the search is part of the analytical procedure. We will return to that issue after constructing the bootstrap.

## Bootstrap uncertainty for a point estimate

Lecture 02a retained the Retail-minus-Horeca difference in medians and observed a difference of 9,706 source monetary units. This observed difference is a **point estimate**: one numerical estimate calculated from the observed sample.

In [ ]:
# Reuse the estimate already calculated from the observed channel samples.
print(f"Point estimate: {observed_difference:,.0f} source monetary units")

Another sample of Retail and Horeca clients would generally produce another point estimate. The **sampling distribution** describes how that estimate would vary if we repeatedly drew new samples from the target Retail and Horeca populations. In practice, however, we normally have only one observed sample from each population.

The **bootstrap** substitutes the observed samples for those unavailable population distributions. Treating each observed sample as an empirical stand-in for its target population, it draws many same-size samples with replacement and recalculates the point estimate each time. The resulting bootstrap distribution approximates the sampling distribution that repeated sampling from the target populations would have produced.

## Empirical distributions and replacement

The bootstrap therefore needs a usable replacement for each unavailable target-population distribution. It constructs that replacement from the observed data.

An **empirical distribution** is a probability model that assigns equal probability to each observed resampling unit—here, each recorded client within its channel. The dataset supplies the observations. The empirical distribution turns them into the possible outcomes from which bootstrap samples are drawn. This case uses one empirical distribution for Retail and another for Horeca.

```text
Unknown population distributions → repeated population samples → sampling distribution
                                       approximated by
Observed samples → empirical distributions → bootstrap samples → bootstrap distribution
```

The related objects have different roles:

| Object | Meaning in this case |
| --- | --- |
| Point estimate | The 9,706-unit difference calculated from the observed clients |
| Empirical distributions | Equal probability on each observed client within its channel |
| Bootstrap sample | One resampled version of the observed dataset, formed by sampling Retail and Horeca clients separately within their channels |
| Bootstrap replicate | The Retail-minus-Horeca median difference calculated from that bootstrap sample |
| Bootstrap distribution | The distribution of bootstrap replicates calculated from many bootstrap samples |

The declared resampling unit is one complete client row. Because the principal statistic uses only `Grocery` and keeps the channels fixed, resampling the two grocery arrays is computationally equivalent to resampling rows within each channel and then selecting `Grocery`.

### Pause: state the empirical-distribution assumption

What must be true enough about the recorded clients for their empirical distributions to provide useful stand-ins for the target Retail and Horeca populations?

##### Answer

The recorded clients must reasonably represent the target channel populations, and the client rows must be independent enough for this resampling purpose. The bootstrap cannot create client types missing from the data or repair an unrepresentative sampling process.

### Replacement in bootstrap samples

Bootstrapping uses sampling with replacement. Let's explore how that differs from sampling without replacement.

Begin with six recorded Retail clients and assign temporary row identities A through F. The identities make it possible to see which rows move, recur, or disappear.

In [ ]:
# Temporary row IDs make duplication and omission visible in the next demonstration.
visible_rows = (
    customers.loc[customers["channel"] == "Retail", ["Grocery"]]
    .head(6)
    .reset_index(drop=True)
    .rename_axis("position")
    .reset_index()
)
visible_rows["row_id"] = list("ABCDEF")
visible_rows[["row_id", "Grocery"]]

Sampling all six rows **without replacement** selects each row once. Sampling six times **with replacement** returns a selected row to the eligible set, so one row can be selected more than once and another can be omitted.

In [ ]:
# One generator advances across both draws; replace controls whether rows can recur.
sampling_rng = np.random.default_rng(7130)

without_replacement = sampling_rng.choice(
    visible_rows["row_id"], size=len(visible_rows), replace=False
)
with_replacement = sampling_rng.choice(
    visible_rows["row_id"], size=len(visible_rows), replace=True
)

pd.DataFrame(
    {
        "without replacement": without_replacement,
        "with replacement": with_replacement,
    }
)

### Pause: inspect replacement

Which row is duplicated in the with-replacement sample, and which row is omitted? Why can an order-insensitive statistic not change when all six rows are sampled without replacement?

##### Answer

Row C appears twice and row B is omitted in the with-replacement sample. The without-replacement sample contains every original row exactly once, so it only changes their order. A median, mean, or proportion calculated from all six values is unchanged by reordering.

Sampling fewer than all available rows without replacement can produce variation, but it represents subsampling from a finite set. The ordinary nonparametric bootstrap instead makes `n` draws with replacement. Each draw then behaves like another draw from the empirical distribution.

## Pseudorandom resampling

Resampling—and many other statistical methods—relies on randomness. A deterministic computer program cannot create true randomness on its own. Statistical software instead uses a **pseudorandom number generator**: a deterministic algorithm whose output is designed to behave like random draws for simulation.

Every pseudorandom number generator needs an initial **state**. A **seed** provides a reproducible way to initialize that state. If no seed is supplied, NumPy can obtain an initial state from the operating system.

Without an explicit seed, the resulting sequence will generally differ each time the code runs. Supplying a seed allows another analyst—or your future self—to reproduce the exact resamples, results, and figures.

Two generators initialized with the same seed begin with the same sequence. A different seed begins a different sequence.

In [ ]:
# Create two matching initial states and one state from a different seed.
first_rng = np.random.default_rng(7130)
different_rng = np.random.default_rng(7131)
matching_rng = np.random.default_rng(7130)

first_draw = first_rng.integers(0, 100, size=5)
different_draw = different_rng.integers(0, 100, size=5)
matching_draw = matching_rng.integers(0, 100, size=5)

print("Seed 7130:", first_draw)
print("Different seed, 7131:", different_draw)
print("Same seed, 7130:", matching_draw)

The identical seeds produce the same initial draw; the different seed produces a different one. The generator is deterministic, while its output is designed to behave like random draws. The seed determines where that reproducible pseudorandom sequence begins.

The generator objects remain available after the first draw. Each additional request advances the generator's internal state and returns the next values in its sequence.

In [ ]:
# Reuse first_rng so that its state advances instead of restarting at the seed.
later_draw = first_rng.integers(0, 100, size=5)

print("First draw:", first_draw)
print("Later draw from the same generator:", later_draw)

An explicit seed is useful when exact reproduction matters—for teaching, debugging, auditing, or rerunning an analysis. It is not required for the generator to run, and it does not make the data representative or the resampling design valid. Initialize the generator once for a procedure and let its state advance.

### Pause: reset or advance the generator

What would happen if code created `np.random.default_rng(7130)` again inside every repetition of a resampling loop? Why should one initialized generator instead be allowed to advance?

##### Answer

Resetting the generator to the same seed inside every repetition restarts the same sequence and can reproduce the same sample. Initializing once and allowing the state to advance produces the repeatable sequence of different draws needed for the bootstrap.

Each procedure below creates its own generator. This keeps its result reproducible without depending on whether an earlier demonstration ran. Within a procedure, the generator is initialized once and its state is allowed to advance.

### One bootstrap replicate

The next cell illustrates a single bootstrap replicate by constructing one same-size, with-replacement sample within each channel and recalculating the median difference. We write this one iteration explicitly so that its mechanics are visible. In the next section, SciPy's `bootstrap` function will automate the same resample-and-recalculate process across many iterations.

In [ ]:
# Initialize once, then draw each channel's original sample size with replacement.
one_sample_rng = np.random.default_rng(7130)

one_retail_sample = one_sample_rng.choice(
    retail_grocery, size=len(retail_grocery), replace=True
)
one_horeca_sample = one_sample_rng.choice(
    horeca_grocery, size=len(horeca_grocery), replace=True
)
one_bootstrap_replicate = median_difference(
    one_retail_sample, one_horeca_sample
)

print(f"Point estimate: {observed_difference:,.0f}")
print(f"One bootstrap replicate: {one_bootstrap_replicate:,.0f}")

The replicate is not a new observed result. It is one value the statistic takes after resampling from the two empirical distributions under the declared design.

## The bootstrap distribution

SciPy now automates the process. It constructs 10,000 bootstrap samples by resampling independently within Retail and Horeca, then calculates the median difference from each one.

In [ ]:
# paired=False resamples the two channel arrays independently.
bootstrap_result = bootstrap(
    (retail_grocery, horeca_grocery),
    statistic=median_difference,
    paired=False,
    n_resamples=10_000,
    confidence_level=0.95,
    method="percentile",
    rng=np.random.default_rng(7130),
)

# SciPy returns both the replicates and summaries computed from them.
bootstrap_differences = bootstrap_result.bootstrap_distribution

pd.Series(
    {
        "Point estimate": observed_difference,
        "Bootstrap standard error": bootstrap_result.standard_error,
    }
).round(1)

This bootstrap is stratified by `Channel` because Retail and Horeca are passed to SciPy as separate arrays. The input structure defines the strata. `paired=False` tells SciPy to resample those arrays independently.

Retail values are drawn only from the Retail empirical distribution, and Horeca values only from the Horeca empirical distribution. Both channels retain their original sample sizes. No value moves from one channel to the other.

The bootstrap standard error estimates how much the median-difference point estimate varies across samples like these. It is calculated from the bootstrap replicates, not directly from the original client values.

### Read the bootstrap distribution

The figure places the point estimate in the bootstrap distribution. Unlike the 02a permutation distribution, this distribution does not represent a no-association model. It represents estimated sample-to-sample variation under the empirical stand-ins and resampling design.

In [ ]:
# Plot the replicates, then locate the estimate calculated from the observed data.
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(bootstrap_differences, bins=45, color="#d9d9d9", edgecolor="white")
ax.axvline(
    observed_difference,
    color="#9e480e",
    linewidth=2.5,
    label=f"Point estimate: {observed_difference:,.0f}",
)
ax.set_title("Bootstrap distribution of the median difference")
ax.set_xlabel("Retail minus Horeca median difference (source monetary units)")
ax.set_ylabel("Number of bootstrap replicates")
ax.legend()
plt.show()

The point estimate lies near the center of the bootstrap distribution. Its visible ridges are expected: medians change at ordered observed values, so many bootstrap samples produce the same or nearby median differences. The distribution need not be smooth or normally shaped to describe the resampled estimate.

The permutation and bootstrap procedures use the same Retail-minus-Horeca median-difference statistic, so their distributions can be compared on one horizontal scale. The histograms below use density rather than raw counts because the two procedures contain different numbers of simulated values.

In [ ]:
# Shared bins and density scaling make the two simulated distributions comparable.
comparison_values = np.concatenate(
    [region_null_differences, bootstrap_differences]
)
comparison_bins = np.linspace(comparison_values.min(), comparison_values.max(), 55)

fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(
    region_null_differences,
    bins=comparison_bins,
    density=True,
    color="#59a14f",
    alpha=0.55,
    label="Region-stratified permutation distribution",
)
ax.hist(
    bootstrap_differences,
    bins=comparison_bins,
    density=True,
    color="#4e79a7",
    alpha=0.55,
    label="Bootstrap distribution",
)
ax.axvline(
    observed_difference,
    color="#9e480e",
    linewidth=2.5,
    label=f"Observed difference: {observed_difference:,.0f}",
)
ax.set_title("The same statistic under two different models")
ax.set_xlabel("Retail minus Horeca median difference (source monetary units)")
ax.set_ylabel("Density")
ax.legend()
plt.show()

The large gap visualizes the same evidence summarized by the small permutation p-value; it is not an independent second result. The permutation procedure breaks the recorded relationship between `Channel` and `Grocery`, so its distribution lies near zero. The bootstrap preserves the two channel-specific empirical distributions, so its distribution lies near the observed difference. The first supports a test of association; the second supports uncertainty estimates such as standard errors and confidence intervals.

## Percentile confidence intervals

The bootstrap distribution now provides a direct route to a confidence interval. A 95% **percentile interval** uses the 2.5th and 97.5th percentiles of the bootstrap replicates as its endpoints. The next cell extracts SciPy's interval and verifies it against those direct quantiles.

In [ ]:
# Extract SciPy's requested percentile interval.
interval_low = bootstrap_result.confidence_interval.low
interval_high = bootstrap_result.confidence_interval.high

# Verify its endpoints from the central 95% of the ordered replicates.
direct_percentiles = np.quantile(bootstrap_differences, [0.025, 0.975])

pd.DataFrame(
    {
        "method": ["SciPy percentile interval", "Direct bootstrap quantiles"],
        "lower endpoint": [interval_low, direct_percentiles[0]],
        "upper endpoint": [interval_high, direct_percentiles[1]],
    }
).round(1)

For populations reasonably represented by these recorded Retail and Horeca clients, the estimated median difference is 9,706 source monetary units. Independent resampling within the two channels gives a 95% percentile bootstrap interval of approximately 8,590 to 11,658.

This interval is one summary of the bootstrap distribution. Its interpretation is conditional on the empirical distributions being useful population stand-ins and on client rows being appropriate independent resampling units.

### Pause: interpret 95% confidence

A colleague says, “There is a 95% probability that the fixed population difference lies between these two computed endpoints.” How would you repair that statement without turning it into a formal derivation?

##### Answer

A precise interpretation is:

> If we repeatedly drew comparable samples from the target Retail and Horeca populations and constructed an interval using this same bootstrap procedure each time, approximately 95% of those intervals would contain the true population median difference.

The interval is what varies. The population difference is treated as fixed, while the calculated interval changes from sample to sample. Therefore, 95% describes the procedure's long-run coverage rate—not the probability that the fixed population difference lies within this particular interval. This interval either contains that difference or it does not.

SciPy's default bootstrap interval is bias-corrected and accelerated, or BCa. This notebook requests `method="percentile"` because the percentile construction can be seen directly in the bootstrap distribution. Other interval methods can produce somewhat different endpoints.

## Resampling structure and stratification

The main bootstrap resamples separately within `Channel`. This is channel-stratified resampling: it preserves both channel sample sizes while allowing the regional composition within each channel to vary across bootstrap samples.

A second procedure preserves more structure by resampling separately within each `Channel`-by-`Region` cell. It fixes all six cell sizes and estimates variation within those recorded cells. The two procedures answer slightly different repeated-sampling questions.

The function below implements that more structured design explicitly. It first divides the grocery values into the six observed `Channel`-by-`Region` cells.

Within each bootstrap iteration, the function draws the original number of clients with replacement from every cell. It then recombines the three regional samples for each channel and calculates one Retail-minus-Horeca median difference. Repeating those operations produces the structured bootstrap distribution while preserving every joint cell count.

In [ ]:
def channel_region_bootstrap_distribution(data, n_resamples, rng):
    """Generate median-difference replicates while fixing joint cell counts."""

    # Build the six strata once. Tuple keys retain both parts of each cell identity,
    # while the dictionary values become the cell-specific empirical distributions.
    cells = {
        (channel, region): group["Grocery"].to_numpy()
        for (channel, region), group in data.groupby(
            ["channel", "Region"], sort=True
        )
    }

    # Preallocate one output position for the statistic from each bootstrap sample.
    differences = np.empty(n_resamples)
    for replicate in range(n_resamples):
        # Draw each cell's original number of values, only from that same cell.
        # replace=True allows observed clients to recur or be omitted.
        sampled_cells = {
            key: rng.choice(values, size=len(values), replace=True)
            for key, values in cells.items()
        }

        # Restore one complete Retail sample from its three regional cell samples.
        sampled_retail = np.concatenate(
            [
                values
                for (channel, _), values in sampled_cells.items()
                if channel == "Retail"
            ]
        )
        # Restore one complete Horeca sample in the same way.
        sampled_horeca = np.concatenate(
            [
                values
                for (channel, _), values in sampled_cells.items()
                if channel == "Horeca"
            ]
        )

        # Each complete structured bootstrap sample yields one replicate.
        differences[replicate] = median_difference(
            sampled_retail, sampled_horeca
        )

    # The collected replicates form the structured bootstrap distribution.
    return differences


# Run the complete procedure with a reproducible sequence of 10,000 samples.
channel_region_differences = channel_region_bootstrap_distribution(
    customers,
    n_resamples=10_000,
    rng=np.random.default_rng(7130),
)
# Construct the same percentile interval used for the main bootstrap.
channel_region_interval = np.quantile(
    channel_region_differences, [0.025, 0.975]
)

# Put both designs on the same scale for a direct sensitivity comparison.
resampling_comparison = pd.DataFrame(
    {
        "resampling structure": [
            "Within Channel",
            "Within Channel × Region",
        ],
        "standard error": [
            bootstrap_result.standard_error,
            channel_region_differences.std(ddof=1),
        ],
        "lower endpoint": [interval_low, channel_region_interval[0]],
        "upper endpoint": [interval_high, channel_region_interval[1]],
    }
)
resampling_comparison.round(1)

Preserving all six `Channel`-by-`Region` cells changes the estimated standard error and interval only slightly. This sensitivity check shows that the uncertainty conclusion changes little under a reasonable alternative structure. It does not prove that one resampling design is universally correct.

### Pause: compare the structural assumptions

Which procedure allows regional composition to vary across samples? Which procedure conditions on the six observed cell sizes? Does the choice materially change the estimated uncertainty in this case?

##### Answer

Resampling within `Channel` allows each channel's regional composition to vary according to its observed mixture. Resampling within `Channel`-by-`Region` conditions on the six cell sizes. The resulting intervals are very similar, so the additional conditioning does not materially change the uncertainty conclusion here.

## Selection and procedural uncertainty

The bootstrap must repeat the **complete analytical procedure** that produced the reported estimate—not merely recalculate its final formula. This matters whenever the procedure includes a choice made from the observed data, such as selecting a variable, model, statistic, or threshold.

This same principle makes the bootstrap broadly useful. It can estimate uncertainty for many classical statistics, user-defined statistics, and multi-step procedures without requiring a separate analytical formula for each one, provided the complete procedure can be reproduced.

For a threshold `t`, classify each client by whether grocery spending exceeds that boundary. The operational statistic is the difference between the resulting Retail and Horeca proportions:

$$
\Delta(t) = P(Grocery > t \mid Retail) - P(Grocery > t \mid Horeca).
$$

If 10,000 is a prespecified operational threshold, it remains fixed in every bootstrap sample. If the analyst instead searches five thresholds and reports the largest absolute difference, the threshold selection must be repeated inside every bootstrap sample.

In [ ]:
# The fixed boundary and the candidate search set represent different procedures.
FIXED_THRESHOLD = 10_000
SEARCH_THRESHOLDS = np.array([5_000, 10_000, 15_000, 20_000, 25_000])


def exceedance_difference(retail, horeca, threshold):
    # Boolean means are proportions above the declared threshold.
    return np.mean(retail > threshold) - np.mean(horeca > threshold)


observed_threshold_effects = np.array(
    [
        exceedance_difference(retail_grocery, horeca_grocery, threshold)
        for threshold in SEARCH_THRESHOLDS
    ]
)
# Select by absolute size so large differences in either direction can win.
observed_winner_position = np.argmax(np.abs(observed_threshold_effects))

pd.DataFrame(
    {
        "threshold": SEARCH_THRESHOLDS,
        "Retail minus Horeca (percentage points)": 100
        * observed_threshold_effects,
        "selected": np.arange(len(SEARCH_THRESHOLDS))
        == observed_winner_position,
    }
).round(1)

The observed search selects 5,000 because it produces the largest absolute exceedance-rate difference among the five declared candidates. That threshold is a result of the search, not a boundary specified independently of these data.

The next function follows both procedures through the same bootstrap samples. One procedure always measures the effect at 10,000. The other recalculates all five candidates and records the winning threshold and its effect in every sample.

In [ ]:
def bootstrap_fixed_and_selected_thresholds(
    retail, horeca, thresholds, fixed_threshold, n_resamples, rng
):
    """Repeat fixed-threshold and threshold-selection procedures together."""

    fixed_effects = np.empty(n_resamples)
    selected_thresholds = np.empty(n_resamples, dtype=int)
    selected_effects = np.empty(n_resamples)

    for replicate in range(n_resamples):
        # Both procedures begin with the same independently resampled channels.
        sampled_retail = rng.choice(retail, size=len(retail), replace=True)
        sampled_horeca = rng.choice(horeca, size=len(horeca), replace=True)

        # The prespecified procedure always evaluates the same boundary.
        fixed_effects[replicate] = exceedance_difference(
            sampled_retail, sampled_horeca, fixed_threshold
        )
        candidate_effects = np.array(
            [
                exceedance_difference(
                    sampled_retail, sampled_horeca, threshold
                )
                for threshold in thresholds
            ]
        )
        # The searched procedure repeats threshold selection in every resample.
        selected_position = np.argmax(np.abs(candidate_effects))
        selected_thresholds[replicate] = thresholds[selected_position]
        selected_effects[replicate] = candidate_effects[selected_position]

    return fixed_effects, selected_thresholds, selected_effects


(
    fixed_threshold_effects,
    selected_thresholds,
    selected_threshold_effects,
) = bootstrap_fixed_and_selected_thresholds(
    retail_grocery,
    horeca_grocery,
    thresholds=SEARCH_THRESHOLDS,
    fixed_threshold=FIXED_THRESHOLD,
    n_resamples=10_000,
    rng=np.random.default_rng(7130),
)

# Normalize the counts to show how often each candidate wins the repeated search.
selection_frequencies = (
    pd.Series(selected_thresholds)
    .value_counts(normalize=True)
    .rename_axis("selected threshold")
    .mul(100)
    .rename("percent of bootstrap samples")
    .sort_index()
)
selection_frequencies.round(1)

The search selects 5,000 in about 96.6% of the bootstrap samples and 10,000 in the remainder. The selected threshold is stable across these declared candidates. That is an empirical result obtained by repeating the selection; it was not guaranteed by the original search.

In [ ]:
# Summarize the resampled output of each complete procedure with percentile bounds.
fixed_interval = np.quantile(fixed_threshold_effects, [0.025, 0.975])
selected_effect_interval = np.quantile(
    selected_threshold_effects, [0.025, 0.975]
)

pd.DataFrame(
    {
        "procedure": [
            "Fixed threshold of 10,000",
            "Select largest effect from five thresholds",
        ],
        "observed effect (percentage points)": [
            100
            * exceedance_difference(
                retail_grocery, horeca_grocery, FIXED_THRESHOLD
            ),
            100 * observed_threshold_effects[observed_winner_position],
        ],
        "lower endpoint": [
            100 * fixed_interval[0],
            100 * selected_effect_interval[0],
        ],
        "upper endpoint": [
            100 * fixed_interval[1],
            100 * selected_effect_interval[1],
        ],
    }
).round(1)

The two rows describe different procedures and therefore different uncertainty questions. The fixed-threshold row estimates uncertainty at a prespecified 10,000-unit boundary. The selected row estimates uncertainty in the effect produced by searching the five candidates and reporting the winner. Holding the observed winner fixed would instead treat 5,000 as though it had been specified in advance.

Repeating selection estimates the behavior of the searched procedure; it does not automatically turn an exploratory winner into an independently confirmed population result. A later sample or held-out data may still be the appropriate next evidence.

## The cumulative evidence record

The record now contains a descriptive point estimate, evidence against a declared no-association model, uncertainty under a declared sampling-resampling model, and a structural sensitivity check.

### Pause: record the uncertainty evidence

Write a short record entry that includes:

1. the point estimate and percentile interval in source monetary units;
2. the resampling unit, strata, and principal assumptions;
3. the result of preserving `Region`; and
4. the next evidence or data improvement that would matter most.

##### Answer

The point estimate is a Retail-minus-Horeca median difference of 9,706 source monetary units. Independent with-replacement resampling within channel gives a 95% percentile interval of approximately 8,590 to 11,658. This interpretation assumes that the empirical channel distributions reasonably represent the target populations and that client rows are appropriate independent resampling units.

Preserving the six `Channel`-by-`Region` cells produces a similar interval, so this structural change does not materially alter the uncertainty conclusion. The most important next improvement is to document or strengthen the sampling frame and collection period before generalizing to a broader population or later time.

### Evidence recorded for this case

| Component | Recorded evidence |
| --- | --- |
| Question and scope | Median annual grocery-spending difference between recorded Retail and Horeca clients |
| Point estimate | Retail minus Horeca: 9,706 source monetary units |
| Null-model evidence | Very unusual under both overall and region-stratified channel-label reassignment |
| Uncertainty evidence | Bootstrap distribution and 95% percentile interval under independent within-channel resampling |
| Sensitivity evidence | Similar interval when resampling within `Channel`-by-`Region` cells |
| Selection status | Prespecified statistics stay fixed; searched thresholds are reselected inside each resample |
| Next check | Improve the population and collection-period evidence or obtain a new confirmatory sample |

The completed record is reproducible because it declares the data, statistic, procedures, resample counts, and seeds. The `Channel`-by-`Region` sensitivity check shows that the principal result changes little when the recorded regional structure is preserved. Broader validity still depends on whether these anonymous clients adequately represent the intended use.

---

Auburn University / Industrial and Systems Engineering<br>
INSY 7130, Pattern Discovery and Time Series Analysis<br>
© Copyright Danny J. O'Leary.

For course materials, attribution, and licensing information, see the
[INSY 7130 course-materials README](../../README.md).